In [1]:
import numpy as np
import math
from collections import Counter

labels = ["Positive", "Negative", "Neutral"]
cm1 = np.array([[100, 5, 15],
                [8, 80, 12],
                [20, 10, 50]])
cm2 = np.array([[106, 4, 10],
                [6, 84, 10],
                [9, 7, 64]])

In [2]:
n = cm1.sum()
correct = np.trace(cm1)
acc = correct / n
se = math.sqrt(acc * (1 - acc) / n)
z = 1.96
lo = acc - z * se
hi = acc + z * se
print("part 1")
print("correct", correct, "of", n)
print("accuracy", round(acc, 4))
print("standard error", round(se, 5))
print("95% CI (normal)", round(lo, 4), round(hi, 4))
denom = 1 + z**2 / n
centre = (acc + z**2 / (2 * n)) / denom
half = z * math.sqrt(acc * (1 - acc) / n + z**2 / (4 * n**2)) / denom
print("95% CI (Wilson)", round(centre - half, 4), round(centre + half, 4))

part 1
correct 230 of 300
accuracy 0.7667
standard error 0.02442
95% CI (normal) 0.7188 0.8145
95% CI (Wilson) 0.7156 0.811


In [3]:
def class_metrics(cm):
    tp = np.diag(cm).astype(float)
    col = cm.sum(axis=0).astype(float)
    row = cm.sum(axis=1).astype(float)
    prec = tp / col
    rec = tp / row
    f1 = 2 * prec * rec / (prec + rec)
    return prec, rec, f1, row

def show(cm):
    prec, rec, f1, support = class_metrics(cm)
    print("class      precision  recall   f1       support")
    for i, l in enumerate(labels):
        print(f"{l:10s} {prec[i]:.4f}     {rec[i]:.4f}   {f1[i]:.4f}   {int(support[i])}")
    macro = f1.mean()
    weighted = (f1 * support).sum() / support.sum()
    micro = np.trace(cm) / cm.sum()
    print("macro F1   ", round(macro, 4))
    print("weighted F1", round(weighted, 4))
    print("micro F1   ", round(micro, 4))
    return prec, rec, f1, macro, weighted, micro

print("parts 2 and 3 (zero-shot)")
r1 = show(cm1)

parts 2 and 3 (zero-shot)
class      precision  recall   f1       support
Positive   0.7812     0.8333   0.8065   120
Negative   0.8421     0.8000   0.8205   100
Neutral    0.6494     0.6250   0.6369   80
macro F1    0.7546
weighted F1 0.7659
micro F1    0.7667


In [4]:
print("part 4 (zero-shot errors)")
for i, l in enumerate(labels):
    row = cm1[i]
    print(l, "row:", row.tolist(), "misclassified", row.sum() - row[i], "of", row.sum())
print("Neutral predicted as Positive:", cm1[2, 0], "and as Negative:", cm1[2, 1])
print("Positive predicted as Neutral:", cm1[0, 2])
print("Negative predicted as Neutral:", cm1[1, 2])
print("errors involving the Neutral class:", cm1[2, 0] + cm1[2, 1] + cm1[0, 2] + cm1[1, 2], "of", n - correct)

part 4 (zero-shot errors)
Positive row: [100, 5, 15] misclassified 20 of 120
Negative row: [8, 80, 12] misclassified 20 of 100
Neutral row: [20, 10, 50] misclassified 30 of 80
Neutral predicted as Positive: 20 and as Negative: 10
Positive predicted as Neutral: 15
Negative predicted as Neutral: 12
errors involving the Neutral class: 57 of 70


In [5]:
print("part 5 (improved prompt)")
n2 = cm2.sum()
acc2 = np.trace(cm2) / n2
se2 = math.sqrt(acc2 * (1 - acc2) / n2)
print("accuracy", round(acc2, 4))
print("95% CI (normal)", round(acc2 - 1.96 * se2, 4), round(acc2 + 1.96 * se2, 4))
r2 = show(cm2)
print()
print("change from stage 1 to stage 2")
print("accuracy   ", round(acc2 - acc, 4))
for i, l in enumerate(labels):
    print(f"{l:10s} precision {r2[0][i] - r1[0][i]:+.4f}  recall {r2[1][i] - r1[1][i]:+.4f}  f1 {r2[2][i] - r1[2][i]:+.4f}")
print("macro F1   ", round(r2[3] - r1[3], 4))
print("weighted F1", round(r2[4] - r1[4], 4))
print("micro F1   ", round(r2[5] - r1[5], 4))

part 5 (improved prompt)
accuracy 0.8467
95% CI (normal) 0.8059 0.8874
class      precision  recall   f1       support
Positive   0.8760     0.8833   0.8797   120
Negative   0.8842     0.8400   0.8615   100
Neutral    0.7619     0.8000   0.7805   80
macro F1    0.8406
weighted F1 0.8472
micro F1    0.8467

change from stage 1 to stage 2
accuracy    0.08
Positive   precision +0.0948  recall +0.0500  f1 +0.0732
Negative   precision +0.0421  recall +0.0400  f1 +0.0410
Neutral    precision +0.1126  recall +0.1750  f1 +0.1435
macro F1    0.0859
weighted F1 0.0812
micro F1    0.08


In [6]:
runs = np.array([0.847, 0.840, 0.853, 0.837, 0.850])
mean = runs.mean()
sd_sample = runs.std(ddof=1)
sd_pop = runs.std(ddof=0)
print("part 6")
print("mean", round(mean, 4))
print("sample std (n-1)", round(sd_sample, 5))
print("population std (n)", round(sd_pop, 5))
print("range", round(runs.max() - runs.min(), 4))
print("coefficient of variation", round(sd_sample / mean * 100, 3), "%")

part 6
mean 0.8454
sample std (n-1) 0.00673
population std (n) 0.00602
range 0.016
coefficient of variation 0.796 %


In [7]:
ref = "the delivery was late but the product quality is good".split()
cand = "the product quality is good but delivery was slow".split()

def ngrams(tokens, k):
    return Counter(tuple(tokens[i:i + k]) for i in range(len(tokens) - k + 1))

def clipped_matches(c, r, k):
    cc = ngrams(c, k)
    rc = ngrams(r, k)
    return sum(min(v, rc[g]) for g, v in cc.items()), sum(cc.values()), sum(rc.values())

print("part 7: BLEU-4")
ps = []
for k in range(1, 5):
    m, tc, tr = clipped_matches(cand, ref, k)
    ps.append(m / tc)
    print(f"{k}-gram: matches {m}, candidate total {tc}, precision {m / tc:.4f}")
c_len = len(cand)
r_len = len(ref)
bp = 1.0 if c_len > r_len else math.exp(1 - r_len / c_len)
bleu = bp * math.exp(sum(math.log(p) for p in ps) / 4)
print("candidate length", c_len, "reference length", r_len)
print("brevity penalty", round(bp, 5))
print("BLEU-4", round(bleu, 4))

part 7: BLEU-4
1-gram: matches 8, candidate total 9, precision 0.8889
2-gram: matches 5, candidate total 8, precision 0.6250
3-gram: matches 3, candidate total 7, precision 0.4286
4-gram: matches 2, candidate total 6, precision 0.3333
candidate length 9 reference length 10
brevity penalty 0.89484
BLEU-4 0.475


In [8]:
def prf(m, tc, tr):
    p = m / tc
    r = m / tr
    f = 2 * p * r / (p + r) if p + r > 0 else 0.0
    return p, r, f

print("ROUGE-1 and ROUGE-2")
for k in (1, 2):
    m, tc, tr = clipped_matches(cand, ref, k)
    p, r, f = prf(m, tc, tr)
    print(f"ROUGE-{k}: matches {m}, precision {p:.4f}, recall {r:.4f}, F1 {f:.4f}")

def lcs_len(a, b):
    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]
    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])
    return dp[-1][-1]

l = lcs_len(cand, ref)
p, r, f = prf(l, len(cand), len(ref))
print(f"ROUGE-L: LCS length {l}, precision {p:.4f}, recall {r:.4f}, F1 {f:.4f}")

ROUGE-1 and ROUGE-2
ROUGE-1: matches 8, precision 0.8889, recall 0.8000, F1 0.8421
ROUGE-2: matches 5, precision 0.6250, recall 0.5556, F1 0.5882
ROUGE-L: LCS length 5, precision 0.5556, recall 0.5000, F1 0.5263


In [9]:
common = Counter(cand) & Counter(ref)
m = sum(common.values())
P = m / len(cand)
R = m / len(ref)
fmean = 10 * P * R / (R + 9 * P)
print("METEOR (exact matches only)")
print("matched unigrams", m, "precision", round(P, 4), "recall", round(R, 4))
print("Fmean", round(fmean, 5))
chunks = 3
penalty = 0.5 * (chunks / m) ** 3
meteor = fmean * (1 - penalty)
print("chunks", chunks, "penalty", round(penalty, 5))
print("METEOR", round(meteor, 4))

METEOR (exact matches only)
matched unigrams 8 precision 0.8889 recall 0.8
Fmean 0.80808
chunks 3 penalty 0.02637
METEOR 0.7868


In [10]:
probs = np.array([0.60, 0.40, 0.50, 0.80, 0.70, 0.30, 0.25, 0.50, 0.20])
logs = np.log(probs)
print("part 8")
for i, (p, lg) in enumerate(zip(probs, logs), 1):
    print(f"token {i}: p = {p:.2f}  ln p = {lg:.4f}")
print("sum of ln p", round(logs.sum(), 4))
print("average negative log likelihood", round(-logs.mean(), 4))
ppl = math.exp(-logs.mean())
print("perplexity", round(ppl, 4))
print("geometric mean of probabilities", round(1 / ppl, 4))

part 8
token 1: p = 0.60  ln p = -0.5108
token 2: p = 0.40  ln p = -0.9163
token 3: p = 0.50  ln p = -0.6931
token 4: p = 0.80  ln p = -0.2231
token 5: p = 0.70  ln p = -0.3567
token 6: p = 0.30  ln p = -1.2040
token 7: p = 0.25  ln p = -1.3863
token 8: p = 0.50  ln p = -0.6931
token 9: p = 0.20  ln p = -1.6094
sum of ln p -7.5929
average negative log likelihood 0.8437
perplexity 2.3249
geometric mean of probabilities 0.4301
